In [0]:
# Read Bronze + Transform
import logging
from pyspark.sql.functions import col, to_date
from pyspark.sql import functions as F
from delta.tables import DeltaTable

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
log = logging.getLogger(__name__)

BRONZE_PATH = "abfss://bronzelayer@cryptodl.dfs.core.windows.net/Batch_Data_Source_1/"
SILVER_PATH = "abfss://silverlayer@cryptodl.dfs.core.windows.net/Batch_Data_Source_1"



log.info("Reading Bronze Source 1...")
bronze_df = spark.read.parquet(BRONZE_PATH)
log.info(f"Bronze rows loaded: {bronze_df.count():,}")

silver_df = (
    bronze_df
    .withColumnRenamed("Name",     "coin_name")
    .withColumnRenamed("Symbol",   "symbol")
    .withColumnRenamed("Open",     "open_price")
    .withColumnRenamed("High",     "high_price")
    .withColumnRenamed("Low",      "low_price")
    .withColumnRenamed("Close",    "close_price")
    .withColumnRenamed("Marketcap","market_cap")
    .withColumn("trade_date", to_date("trade_date"))
    .drop("SNo", "Date")
)

log.info("Transformation complete")


In [0]:
# CELL 2 — Data Quality Report

log.info("Running data quality checks...")

total_rows       = silver_df.count()
null_symbols     = silver_df.filter(col("symbol").isNull()).count()
null_dates       = silver_df.filter(col("trade_date").isNull()).count()
negative_prices  = silver_df.filter(col("close_price") < 0).count()
duplicates       = (
    silver_df.groupBy("symbol", "trade_date")
             .count()
             .filter(col("count") > 1)
             .count()
)
bad_high_low     = silver_df.filter(col("high_price") < col("low_price")).count()

log.info("=" * 50)
log.info("SOURCE 1 SILVER QUALITY REPORT")
log.info("=" * 50)
log.info(f"Total Rows        : {total_rows:,}")
log.info(f"Null Symbols      : {null_symbols}")
log.info(f"Null Trade Dates  : {null_dates}")
log.info(f"Negative Prices   : {negative_prices}")
log.info(f"Duplicate Records : {duplicates}")
log.info(f"Invalid High/Low  : {bad_high_low}")
log.info("=" * 50)

if null_symbols == 0 and null_dates == 0 and negative_prices == 0 and duplicates == 0 and bad_high_low == 0:
    log.info("ALL SILVER TESTS PASSED")
else:
    log.warning("DATA QUALITY ISSUES FOUND")

display(silver_df.limit(20))


In [0]:
# CELL 3 — Write Silver Delta (Create or Merge)

log.info(f"Writing Silver Delta to: {SILVER_PATH}")

if not DeltaTable.isDeltaTable(spark, SILVER_PATH):
    (
        silver_df.write
        .format("delta")
        .mode("overwrite")
        .save(SILVER_PATH)
    )
    log.info("Silver Delta Table Created")

else:
    (
        DeltaTable.forPath(spark, SILVER_PATH)
        .alias("target")
        .merge(
            silver_df.alias("source"),
            "target.symbol = source.symbol AND target.trade_date = source.trade_date"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    log.info("Silver Delta Table Merged Successfully")


In [0]:
# CELL 4 — Verify
log.info("Verifying Silver Delta...")
df_verify = spark.read.format("delta").load(SILVER_PATH)

total    = df_verify.count()
symbols  = df_verify.select("symbol").distinct().count()
date_rng = df_verify.agg(F.min("trade_date"), F.max("trade_date")).collect()[0]

log.info(f"Total rows    : {total:,}")
log.info(f"Unique symbols: {symbols}")

log.info(f"Date range    : {date_rng[0]} → {date_rng[1]}")

display(df_verify.limit(20))